In [1]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors  
from molpher.core import ExplorationTree as ETree

class NitroReduction(MorphingOperator):
    def __init__(self):
        super(NitroReduction, self).__init__()
        self._name = "Nitro Reduction (Phase I - Safe)"
        self._target_nitrogens = []
        self.NITRO_PATTERN = Chem.MolFromSmarts("[C,c][N;X3](=[O,O-])~[O,O-]")

    def setOriginal(self, mol):
        super(NitroReduction, self).setOriginal(mol)
        self._target_nitrogens = []

        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        matches = rdkit_mol.GetSubstructMatches(self.NITRO_PATTERN)
        for match in matches:
            if match[1] not in self._target_nitrogens:
                self._target_nitrogens.append(match[1])

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._target_nitrogens:
            return MolpherMol(other=rdkit_mol)

        n_idx = random.choice(self._target_nitrogens)

        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            n_atom = rw_mol.GetAtomWithIdx(n_idx)
            
            o_indices = [neighbor.GetIdx() for neighbor in n_atom.GetNeighbors() if neighbor.GetAtomicNum() == 8]
            
            for o_idx in o_indices:
                bond = rw_mol.GetBondBetweenAtoms(n_idx, o_idx)
                if bond:
                    rw_mol.RemoveBond(n_idx, o_idx)
            
            n_atom.SetFormalCharge(0)
            n_atom.SetNoImplicit(False)
            n_atom.SetNumExplicitHs(2)
            for prop in list(n_atom.GetPropNames()):
                n_atom.ClearProp(prop)
            
            new_mol = rw_mol.GetMol()
            new_mol.UpdatePropertyCache(strict=False)
            
            atoms_to_remove = []
            for atom in new_mol.GetAtoms():
                if atom.GetAtomicNum() == 8 and atom.GetDegree() == 0:
                    atoms_to_remove.append(atom.GetIdx())

            edit = Chem.RWMol(new_mol)
            for idx in sorted(atoms_to_remove, reverse=True):
                edit.RemoveAtom(idx)
            new_mol = edit.GetMol()

            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            return MolpherMol(other=new_mol)

        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name

nitro_op = NitroReduction()

test_nitro_molecules = {
    "1. Νιτροβενζόλιο (Αρωματική Nitro -> Ανιλίνη)": "O=[N+]([O-])C1=CC=CC=C1",
    "2. 2-Νιτροπροπάνιο (Αλειφατική Nitro -> Αμίνη)": "CC(C)[N+](=O)[O-]",
    "3. Ανιλίνη (Πρωτοταγής Αμίνη -> Πρέπει να αγνοηθεί πλήρως)": "NC1=CC=CC=C1"
}

print("=== STARTING NITRO REDUCTION TESTING ===")
for name, smiles in test_nitro_molecules.items():
    mol = MolpherMol(smiles)
    nitro_op.setOriginal(mol)
    product = nitro_op.morph()
    
    print(f"\n{name}")
    print(f"  SOURCE: {mol.getSMILES()}")
    print(f"  TARGET: {product.getSMILES() if product and product.getSMILES() != mol.getSMILES() else 'No change (Safe)'}")
print("\n========================================")

=== STARTING NITRO REDUCTION TESTING ===

1. Νιτροβενζόλιο (Αρωματική Nitro -> Ανιλίνη)
  SOURCE: O=[N+]([O-])C1=CC=CC=C1
  TARGET: NC1=CC=CC=C1

2. 2-Νιτροπροπάνιο (Αλειφατική Nitro -> Αμίνη)
  SOURCE: CC(C)[N+](=O)[O-]
  TARGET: CC(C)N

3. Ανιλίνη (Πρωτοταγής Αμίνη -> Πρέπει να αγνοηθεί πλήρως)
  SOURCE: NC1=CC=CC=C1
  TARGET: No change (Safe)

